# Linha de impedimento no Google Colab

Este notebook roda o projeto inteiro no Colab: exporta o YOLO de jogadores para ONNX, treina (fine-tuning em PyTorch) o detector de bola, e abre a interface de análise numa célula, com a detecção rodando aqui no Colab.

**Antes de começar:** Ambiente de execução → Alterar o tipo de ambiente de execução → **GPU T4**.

**O que fica salvo:** o Colab apaga os arquivos quando a sessão termina. Por isso modelos, treinos e imagens exportadas vão para a pasta `offside-detector-colab` no seu Google Drive. Nas próximas vezes, a etapa 3 detecta o que já existe e pula.

| Etapa | Quando rodar |
|---|---|
| 1 e 2. Ambiente | Toda vez que abrir o notebook |
| 3. Modelo de jogadores | Uma vez |
| 4. Treino da bola | Uma vez (ou quando quiser treinar de novo) |
| 5. Análise | Sempre que for analisar um lance |
| 6. Rastrear a bola num vídeo | Opcional |

## 1. Preparar o ambiente

Monta o Google Drive e prepara o projeto. Na primeira vez, o Colab pede o arquivo `offside-detector.zip`; ele fica guardado no Drive e não é pedido de novo.

In [1]:
#@title 1. Montar o Drive e preparar o projeto
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/offside-detector-colab')   # tudo que precisa sobreviver entre sessões
for sub in ('models', 'runs', 'outputs', 'videos'):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

PROJECT = Path('/content/offside-detector')
if not PROJECT.exists():
    zip_path = BASE / 'offside-detector.zip'
    if not zip_path.exists():
        from google.colab import files
        print('Envie o offside-detector.zip (ele fica guardado no Drive para as próximas vezes).')
        enviado = files.upload()
        shutil.move(next(iter(enviado)), zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')

# a pasta models/ do projeto passa a ser a do Drive: nada exportado ou treinado se perde
models = PROJECT / 'models'
if models.exists() and not models.is_symlink():
    shutil.rmtree(models)
if not models.exists():
    models.symlink_to(BASE / 'models')
os.chdir(PROJECT)
print('Projeto:', PROJECT)
print('Modelos, treinos e imagens ficam em:', BASE)

Mounted at /content/drive
Envie o offside-detector.zip (ele fica guardado no Drive para as próximas vezes).


Saving offside-detector.zip to offside-detector.zip
Projeto: /content/offside-detector
Modelos, treinos e imagens ficam em: /content/drive/MyDrive/offside-detector-colab


In [2]:
#@title 2. Instalar as dependências (1 a 2 minutos)
!pip uninstall -q -y onnxruntime
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime-gpu
!pip install -q -e . --no-deps

import sys
sys.path.insert(0, str(PROJECT / 'src'))
import torch, onnxruntime as ort
if hasattr(ort, 'preload_dlls'):
    ort.preload_dlls()   # usa o CUDA/cuDNN que já vem com o PyTorch do Colab
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', gpu or 'nenhuma. Troque o ambiente de execução para GPU T4, ou o treino vai levar horas.')
print('ONNX Runtime', ort.__version__, ort.get_available_providers())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.1

## 3. Modelo de jogadores

YOLO11 pré-treinado no COCO (a classe "pessoa" serve para os jogadores), exportado para ONNX com entrada de 1280 px, porque jogadores em plano aberto são pequenos.

In [3]:
#@title 3. Exportar o YOLO de jogadores para ONNX
if (BASE / 'models' / 'players.onnx').exists():
    print('models/players.onnx já existe no Drive, pulando.')
else:
    !python scripts/export_onnx.py --weights yolo11s.pt --out models/players.onnx --imgsz 1280

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.161 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 87.8 GFLOPs

PyTorch: starting from 'yolo11s.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 84, 33600) (18.4 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 8.1s, saved as 'yolo11s.onnx' (36.8 MB)

Export complete (13.0s)
Re

## 4. Treino do detector de bola

Fine-tuning de um YOLO pré-treinado só na classe "bola", com o dataset público do Roboflow.

**Chave do Roboflow:** crie uma conta grátis em roboflow.com, copie a API key (Settings → API Keys) e guarde no Colab em 🔑 **Secrets** (barra lateral esquerda) com o nome `ROBOFLOW_API_KEY`, liberando o acesso para este notebook.

In [4]:
#@title 4a. Baixar o dataset da bola
WORKSPACE = "roboflow-jvuqo"                 #@param {type:"string"}
PROJETO_ROBOFLOW = "football-ball-detection-rejhg"  #@param {type:"string"}
VERSAO = 1                                   #@param {type:"integer"}

from google.colab import userdata
try:
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    raise SystemExit('Não achei o segredo ROBOFLOW_API_KEY. Crie em 🔑 Secrets, na barra lateral, e libere o acesso ao notebook.')

!python scripts/download_ball_dataset.py --workspace {WORKSPACE} --project {PROJETO_ROBOFLOW} --version {VERSAO} --out data/ball_raw

# o fatiamento só faz sentido com imagens na resolução original
import cv2, glob, collections
tamanhos = collections.Counter(cv2.imread(f).shape[1::-1] for f in sorted(glob.glob('data/ball_raw/train/images/*'))[:60])
print('Resoluções (largura, altura):', dict(tamanhos))
if max(max(t) for t in tamanhos) <= 640:
    print('ATENÇÃO: as imagens já vêm reduzidas para 640 px. Escolha outra versão do dataset, sem "Resize", '
          'ou a bola vai ficar pequena demais para aprender.')

loading Roboflow workspace...
loading Roboflow project...

Extracting Dataset Version Zip to data/ball_raw in yolov8:: 100% 1818/1818 [00:00<00:00, 3862.31it/s]
Dataset salvo em /content/offside-detector/data/ball_raw
Resoluções (largura, altura): {(1920, 1080): 60}


In [5]:
#@title 4b. Fatiar o dataset na escala do SAHI
!rm -rf data/ball_tiles
!python scripts/tile_dataset.py data/ball_raw data/ball_tiles --tile 640 --blur-prob 0.3

train: 1864 tiles com bola, 412 sem bola
valid: 173 tiles com bola, 46 sem bola
data.yaml escrito em data/ball_tiles/data.yaml


In [6]:
#@title 4c. Treinar (a barra de progresso mostra o tempo por época)
EPOCAS = 60                 #@param {type:"integer"}
MODELO_BASE = "yolo11s.pt"  #@param ["yolo11n.pt", "yolo11s.pt", "yolov8s.pt"]

!python scripts/train_ball.py --data data/ball_tiles/data.yaml --model {MODELO_BASE} --epochs {EPOCAS} --project "{BASE / 'runs'}" --name ball --device 0

Ultralytics 8.4.161 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data/ball_tiles/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ball, nbs=64, nms=None, opset=None, optimize=False,

**Se a sessão cair no meio do treino:** rode de novo as etapas 1, 2, 4a e 4b (o dataset fica fora do Drive, porque treinar lendo do Drive é lento) e depois a célula abaixo, que continua de onde parou a partir do último checkpoint salvo no Drive.

In [ ]:
#@title 4d. Continuar um treino interrompido
import glob
ultimos = sorted(glob.glob(str(BASE / 'runs' / '*' / 'weights' / 'last.pt')), key=os.path.getmtime)
if not ultimos:
    raise SystemExit('Nenhum treino salvo no Drive ainda.')
print('Continuando de', ultimos[-1])
!python scripts/train_ball.py --data data/ball_tiles/data.yaml --resume "{ultimos[-1]}"

In [7]:
#@title 4e. Exportar a bola para ONNX e validar o modelo exportado
import glob
melhores = sorted(glob.glob(str(BASE / 'runs' / '*' / 'weights' / 'best.pt')), key=os.path.getmtime)
if not melhores:
    raise SystemExit('Nenhum best.pt encontrado. Rode o treino (4c) antes.')
print('Usando', melhores[-1])
!python scripts/export_onnx.py --weights "{melhores[-1]}" --out models/ball.onnx --dynamic
# mesmas métricas do .pt, agora medidas no .onnx: confirma que a exportação não perdeu nada
!yolo val model=models/ball.onnx data=data/ball_tiles/data.yaml imgsz=640 batch=1

Usando /content/drive/MyDrive/offside-detector-colab/runs/ball/weights/best.pt
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.161 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/offside-detector-colab/runs/ball/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (18.3 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 3.9s, saved as '/content/drive/MyDrive/offside-detector-colab/runs/ball/weights/best.onnx' (36.3 MB)

Export complete (5.8s)
Results saved to /content/drive/MyDrive/offside-detector-colab/runs/ball/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/offside-detector-colab/runs/ball/weights/best.onnx imgsz=640 
Validate:       

## 5. Análise do lance

A interface abre abaixo. Abra o vídeo (ou a imagem do toque) pelo botão dela: o arquivo vem do seu computador. O fluxo é o mesmo do site, com dois botões a mais na etapa 3: **Detectar jogadores** e **Detectar a bola**, que mandam o frame do toque para o YOLO em ONNX rodando aqui no Colab.

Com jogadores detectados, o 1º clique de Defensor ou Atacante pode ser na caixa do jogador; **Tab** alterna entre caixas sobrepostas e **Descartar (R)** tira o árbitro. **Salvar imagem** grava o PNG em `offside-detector-colab/outputs` no Drive.

Clique na imagem antes de usar os atalhos de teclado: assim eles vão para a interface, e não para o notebook.

In [8]:
#@title 5. Abrir a interface de análise
from offside.colab import launch
bridge = launch(models_dir=BASE / 'models', out_dir=BASE / 'outputs')

## 6. (Opcional) Rastrear a bola num vídeo inteiro

Roda o detector de bola com SAHI em todos os frames, filtra com o Kalman e interpola os buracos. Gera o `.ball.json` (que a versão desktop usa) e um vídeo com a bola marcada, para você ver o tracking funcionando: círculo laranja onde a bola foi detectada, amarelo onde foi interpolada.

In [ ]:
#@title 6. Rastrear a bola e gerar o vídeo marcado
import json, subprocess, cv2
from google.colab import files

enviado = files.upload()
nome = next(iter(enviado))
video = BASE / 'videos' / nome
shutil.move(nome, video)
!python scripts/track_ball.py "{video}"

track = json.loads(video.with_suffix('.ball.json').read_text())['frames']
cap = cv2.VideoCapture(str(video))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
tmp = '/content/bola_tmp.avi'
out = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'MJPG'), fps, (w, h))
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    s = track.get(str(i))
    if s:
        cor = (26, 159, 255) if s['source'] == 'det' else (0, 230, 255)
        cv2.circle(frame, (round(s['x']), round(s['y'])), round(max(s['r'], 6)) + 6, cor, 2)
    out.write(frame)
    i += 1
out.release()
destino = BASE / 'outputs' / f'{video.stem}_bola.mp4'
subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', tmp, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(destino)], check=True)
print('Vídeo com a bola marcada:', destino)